# AICONS Lab Functional Gradient Workflow
## Read before beginning
Before running anything in this notebook, generate connectomes for each subject using a method of your choosing and save them as a `csv` with the first column being region labels. Before beginning, you will need:
1. A text file containing paths to each connectome you have generated.
2. A `tsv` lookup table containing the following information about each parcellation in your connectomes: `index`, `name`, `color`, and `network`. See the [regions](regions.tsv) file for an example. 
## Environment
This notebook is designed to work with [uv](https://docs.astral.sh/uv/getting-started/installation/). If you do not wish to use `uv`, create your own environment based on the pyproject.toml file.

To create the environment with `uv`, simply run `uv venv`, followed by `uv sync`. If using vscode, you may need to reload the window for it to find the newly created python environment.

Since `plotly` requires an installation of chrome, run the following command `uv run plotly_get_chrome -y` if using `uv` or `plotly_get_chrome -y` with the environment activated.

## Functions - Does not produce outputs

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from brainspace.datasets import load_conte69, load_group_fc, load_parcellation
from brainspace.gradient import GradientMaps
from brainspace.plotting import plot_hemispheres
from brainspace.utils.parcellation import map_to_labels
from brainspace.vtk_interface.wrappers.data_object import BSPolyData
from nilearn import plotting
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm


def get_fisher(arr: np.ndarray) -> np.ndarray:
    eye = np.eye(*arr.shape).astype(bool)
    arr[eye] = 0
    arr = np.arctanh(arr)
    arr[eye] = 1
    return arr


def get_inv_fisher(arr: np.ndarray) -> np.ndarray:
    eye = np.eye(*arr.shape).astype(bool)
    arr[eye] = 0
    arr = np.tanh(arr)
    arr[eye] = 1
    return arr


def plot_connectome(
    connectome: np.ndarray, lut: pd.DataFrame, save_path: Path, prefix: str
) -> None:
    labels = lut["name"].tolist()
    plotting.plot_matrix(
        connectome,
        figure=(15, 15),
        labels=labels,
        vmax=0.8,
        vmin=0,
        reorder=False,
        cmap="viridis",
    )
    plt.title("Original connectome", fontsize=16, pad=20)
    plt.savefig(save_path.joinpath(f"{prefix}connectome.png"))
    plt.close()
    similarities = cosine_similarity(connectome, dense_output=False)
    plotting.plot_matrix(
        similarities,
        figure=(15, 15),
        labels=labels,
        vmax=0.8,
        vmin=0,
        reorder=True,
        cmap="viridis",
    )
    plt.title("Cosine similarity matrix of schafer", pad=20)
    plt.savefig(save_path.joinpath(f"{prefix}connectome_cosine.png"))
    plt.close()


def plot_gradient_features(
    gm: GradientMaps, save_path: Path, prefix: str
) -> None:
    fig, ax = plt.subplots(1, figsize=(5, 4))
    ax.scatter(range(gm.lambdas_.size), gm.lambdas_)  # type: ignore
    ax.set_xlabel("Gradient")
    ax.set_ylabel("Eigenvalue")
    ax.set_title("Eigenvalues for first 10 extracted components of schafer")
    plt.savefig(save_path.joinpath(f"{prefix}gradient_eigenvalues.png"))
    plt.close()

    total_egnvalues = sum(abs(gm.lambdas_))  # type: ignore
    var_exp = [
        (i / total_egnvalues * 100)
        for i in sorted(abs(gm.lambdas_), reverse=True)  # type: ignore
    ]
    plt.bar(range(0, len(var_exp)), var_exp, alpha=0.5, align="center")
    plt.ylabel("Explained variance (%)", fontsize=15)
    plt.xlabel("Component (gradient) number", fontsize=15)
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)
    plt.title("Variance explained:", fontsize=17)
    plt.tight_layout()
    plt.savefig(save_path.joinpath(f"{prefix}explained_variance.png"))
    plt.close()


def plot_gradient_space(
    gradients: np.ndarray,
    lut: pd.DataFrame,
    include_networks: list | None,
    save_path: Path,
    prefix: str,
) -> None:

    df = px.data.tips()
    if include_networks is not None:
        mask = lut["Network"].isin(include_networks).to_numpy()
        regions = lut[lut["Network"].isin(include_networks)]
        regions = regions["Network"].tolist()
    else:
        mask = np.ones(shape=lut.shape[0]).astype(bool)
        regions = lut["Network"].tolist()
    fig = px.scatter(
        df,
        x=gradients[mask, 1],  # type: ignore
        y=gradients[mask, 0],  # type: ignore
        color=regions,
        color_continuous_scale=px.colors.sequential.Viridis,
        title="Gradient 1 vs 2 for schafer",
        labels=dict(y="Gradient 1", x="Gradient 2", color="regions"),
    )
    fig.update_traces(marker=dict(size=30))
    fig.update_layout(
        title_font_size=24,
        xaxis_title_font_size=20,
        yaxis_title_font_size=20,
        legend_title_font_size=20,
        legend_font_size=18,
    )
    pio.write_image(
        fig,
        save_path.joinpath(f"{prefix}principal_gradients.png"),
        width=1250,
        height=1250,
        scale=2,
    )


def generate_gradients(
    conn: np.ndarray,
    gradient_map_kwargs: dict[str, int | str],
    sparsity: float,
    lut: pd.DataFrame,
    include_networks: list[str] | None,
    save_path: Path,
    template_gm: GradientMaps | None = None,
    n_iters: int = 10,
) -> None:

    prefix = ""
    if template_gm is not None:
        assert "alignment" in gradient_map_kwargs, (
            "If using template, you must specify an alignment strategy"
        )
        template_gm = template_gm.gradients_  # type: ignore
        prefix = "aligned_"

    gm = GradientMaps(**gradient_map_kwargs)  # type: ignore
    gm.fit(
        conn,
        reference=template_gm,
        sparsity=sparsity,
        n_iter=n_iters,
    )

    plot_connectome(conn, lut, save_path, prefix)
    plot_gradient_features(gm, save_path, prefix)
    if gm.aligned_ is not None:
        plot_gradient_space(
            gm.aligned_,  # type: ignore
            lut,
            include_networks,
            save_path,
            prefix,  # type: ignore
        )
        np.savetxt(
            save_path.joinpath("aligned_gradients.csv"),
            gm.aligned_,  # type:ignore
            delimiter=",",
        )
    else:
        plot_gradient_space(
            gm.gradients_,  # type: ignore
            lut,
            include_networks,
            save_path,
            prefix,  # type: ignore
        )
        np.savetxt(
            save_path.joinpath("gradients.csv"),
            gm.gradients_,  # type:ignore
            delimiter=",",
        )


def plot_brain_hemis(
    gradients: np.ndarray,
    surf_lh: BSPolyData,
    surf_rh: BSPolyData,
    surf_labelling: np.ndarray,
    num: int,
    save_path: Path,
    screenshot: bool,
    transparent: bool,
    interactive: bool,
):
    filename = None
    to_plot = []
    grad_labels = []
    for i in range(num):
        to_plot.append(
            map_to_labels(
                gradients[:, i],
                surf_labelling,
                mask=surf_labelling != 0,
                fill=np.nan,  # type: ignore
            )
        )
        grad_labels.append(f"Gradient {i + 1}")

    if not interactive:
        filename = save_path.parent.joinpath("grads_on_brain.png")
    plot_hemispheres(
        surf_lh,
        surf_rh,
        array_name=to_plot,
        size=(1250, 400),
        cmap="viridis_r",
        color_bar=True,
        label_text=grad_labels,
        zoom=1.55,
        screenshot=screenshot,
        interactive=interactive,
        transparent_bg=transparent,
        filename=filename,
    )


# Generating Individual Gradients - Unaligned 
This workflow will generate gradients for each subject in their native gradient space (unaligned).

In [ ]:
# Enter paths to your subject text file and lookup table below
lut_path = ""
subjects = ""
results = Path("")
num_regions = 100
################ Default Settings, edit if you want ##############
gradient_map_kwargs = {
    "n_components": 10,
    "approach": "dm",
    "random_state": 0,
    "kernel": "cosine",
}
sparsity = 0.8  # Proportion of edges to retain in the affinity matrix (0.8 means keep top 20% of edges)
lut = pd.read_csv(lut_path, index_col=0, delimiter="\t")
numbers_array = np.arange(1, num_regions + 1)
include_networks = None  # Change to a list of networks to include in plots

# Gradient generation below
with open(subjects, "r") as f:
    sub_list = f.readlines()
subs = [Path(x.removesuffix("\n")) for x in sub_list]


for path in tqdm(subs):
    save_path = results.joinpath(path.name.split(".")[0])
    if save_path.joinpath("gradients.csv").is_file():
        continue
    conn = pd.read_csv(path, index_col=0).to_numpy()
    if conn.shape[0] != conn.shape[1]:
        raise ValueError(
            "Ensure data is saved with the first column being region lut"
        )
    save_path = results.joinpath(path.name.split(".")[0])
    save_path.mkdir(parents=True, exist_ok=True)
    generate_gradients(
        conn,
        gradient_map_kwargs,
        sparsity,
        lut,
        include_networks,
        save_path,
    )

# Generating Aligned Gradients
This workflow will generate gradients for each subject aligned to a template gradient. By default, this script uses the Schaefer aligned template connectome from the Human Connectome Project, available through the `Branspace` python package - see [here]( https://brainspace.readthedocs.io/en/latest/generated/brainspace.datasets.load_group_fc.html#brainspace.datasets.load_group_fc ) for more details. If you wish to use your own template connectome, input a path to the template connectome and ensure the first column of the `csv` file is the region labels.

In [ ]:
# Enter paths to your subject text file and lookup table below
lut_path = ".tsv"
subjects = ".txt"
results = Path("")
num_regions = 100
template_connectome_path = ""
average = True
################ Default Settings, edit if you want ##############
template_gradient_map_kwargs = {
    "n_components": 10,
    "approach": "dm",
    "random_state": 0,
    "kernel": "cosine",
}
gradient_map_kwargs = {
    "n_components": 10,
    "approach": "dm",
    "random_state": 0,
    "kernel": "cosine",
    "alignment": "procrustes",
}
n_iters = 100  # Maximum number of iterations for procrustes alignment (only relevant if alignment is set to procrustes)
template_sparsity = 0.8  # Proportion of edges to retain in the affinity matrix (0.8 means keep top 20% of edges)
dataset_sparsity = 0.8  # Proportion of edges to retain in the affinity matrix (0.8 means keep top 20% of edges)
lut = pd.read_csv(lut_path, index_col=0, delimiter="\t")
numbers_array = np.arange(1, num_regions + 1)
include_networks = None  # Change to a list of networks to exclude from plots

if not template_connectome_path:
    template_connectome = load_group_fc("schaefer", num_regions)
else:
    # Gradient generation below
    template_connectome = pd.read_csv(
        template_connectome_path, index_col=0
    ).to_numpy()
if template_connectome.shape[0] != template_connectome.shape[1]:
    raise ValueError(
        "Ensure the first column of the template contains the region labels"
    )

template_gm = GradientMaps(**template_gradient_map_kwargs)
template_gm.fit(template_connectome, sparsity=template_sparsity)
results.mkdir(parents=True, exist_ok=True)
plot_connectome(template_connectome, lut, results, "template_")
plot_gradient_features(template_gm, results, "template_")
plot_gradient_space(
    template_gm.gradients_,  # type: ignore
    lut,
    include_networks,
    results,
    "template_",
)
with open(subjects, "r") as f:
    sub_list = f.readlines()
subs = [Path(x.removesuffix("\n")) for x in sub_list]

conns = []
for path in tqdm(subs):
    conn = pd.read_csv(path, index_col=0).to_numpy()
    if conn.shape[0] != conn.shape[1]:
        raise ValueError(
            "Ensure data is saved with the first column being region name"
        )
    if average:
        conns.append(get_fisher(conn))
        continue

    save_path = results.joinpath(path.name.split(".")[0])
    save_path.mkdir(parents=True, exist_ok=True)
    generate_gradients(
        conn,
        gradient_map_kwargs,
        dataset_sparsity,
        lut,
        include_networks,
        save_path,
        template_gm,
        n_iters,
    )

if average:
    conn = get_inv_fisher(np.array(conns).mean(axis=0))
    generate_gradients(
        conn,
        gradient_map_kwargs,
        dataset_sparsity,
        lut,
        include_networks,
        results,
        template_gm,
        n_iters,
    )


# Plotting Gradients on Brain Surface
This workflow is designed to run **after** gradients have been generated for each subject. Input a `.txt` file with each line being a path to a `*gradients.csv` file as generated by one of the two prior workflows. It is currently only configured to project the gradients onto the `conte69` surface available through `Brainspace` see [here](https://brainspace.readthedocs.io/en/latest/generated/brainspace.datasets.load_conte69.html#brainspace.datasets.load_conte69) for more details. Along with this, it is currently only configured to map the Schaefer atlas parcelation to the conte surface.

In [ ]:
# Enter paths to your subject text file and lookup table below
gradients = ".txt"
results = Path("")
num_regions = 100
num_gradients_to_plot = 3
interactive = False


screenshot = True
transparent = True
filename = None
if interactive:
    screenshot = False
    transpartent = False

with open(gradients, "r") as f:
    sub_list = f.readlines()
subs = [Path(x.removesuffix("\n")) for x in sub_list]

labeling = load_parcellation("schaefer", num_regions, join=True)
surf_lh, surf_rh = load_conte69()  # type: ignore
for path in subs:
    gradients = np.loadtxt(path, delimiter=",")
    plot_brain_hemis(
        gradients,
        surf_lh,  # type: ignore
        surf_rh,  # type: ignore
        labeling,  # type: ignore
        num_gradients_to_plot,
        path,
        screenshot,
        transparent,
        interactive,
    )